In [8]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings("ignore")

In [9]:
llm_model = "gpt-3.5-turbo"

In [10]:
#Setup
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model=llm_model, temperature=0)

In [11]:
#Define Tools
import wikipedia

@tool
def calculator(expression: str) -> str:
    """Useful for math calculations"""
    try:
        expression = expression.lower().strip()

        # Handle "x% of y"
        if "of" in expression and "%" in expression:
            parts = expression.replace("%", "").split("of")
            num = float(parts[0].strip())
            total = float(parts[1].strip())
            return str((num / 100) * total)

        # Handle normal %
        expression = expression.replace("%", "/100")

        return str(eval(expression))

    except Exception as e:
        return f"Error: {str(e)}"


@tool
def wiki_search(query: str) -> str:
    """Search Wikipedia"""
    try:
        return wikipedia.summary(query, sentences=2)
    except:
        return "No result"

In [12]:
#Create Agent
llm = ChatOpenAI(model=llm_model, temperature=0)

tools = [calculator, wiki_search]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("human", "{input}")
])

llm_with_tools = llm.bind_tools(tools)

In [13]:
#Math Example
response = llm_with_tools.invoke("What is 25% of 300?")
if response.tool_calls:
    tool_call = response.tool_calls[0]

    tool_name = tool_call["name"]
    args = tool_call["args"]

    tool_map = {t.name: t for t in tools}

    tool_result = tool_map[tool_name].invoke(args["expression"])

    print("Tool Result:", tool_result)

Tool Result: 75.0


In [15]:
#Wikipedia Example
question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU)\
what book did he write?"
result = llm_with_tools.invoke(question)

if result.tool_calls:
    tool_call = result.tool_calls[0]

    tool_name = tool_call["name"]
    args = tool_call["args"]code

    tool_map = {t.name: t for t in tools}

    tool_result = tool_map[tool_name].invoke(args["query"])

    print("Tool Result:", tool_result)

else:
    print(result.content)

Tool Result: Tom Michael Mitchell (born August 9, 1951) is an American computer scientist and the Founders University Professor at Carnegie Mellon University (CMU). He is a founder and former chair of the Machine Learning Department at CMU. Mitchell is known for his contributions to the advancement of machine learning, artificial intelligence, and cognitive neuroscience and is the author of the textbook Machine Learning.


In [16]:
@tool
def python_repl(code: str) -> str:
    """Executes python code"""
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return str(e)

In [17]:
tools = [python_repl]

llm_with_tools = llm.bind_tools(tools)

In [18]:
customer_list = [
    ["Harrison", "Chase"], 
    ["Lang", "Chain"],
    ["Dolly", "Too"],
    ["Elle", "Elem"], 
    ["Geoff","Fusion"], 
    ["Trance","Former"],
    ["Jen","Ayai"]
]

In [19]:
result = llm_with_tools.invoke(f"""Sort these customers by last name and then first name 
    and print the output: {customer_list}""")
if result.tool_calls:
    tool_call = result.tool_calls[0]

    tool_name = tool_call["name"]
    args = tool_call["args"]

    tool_map = {t.name: t for t in tools}

    tool_result = tool_map[tool_name].invoke(args["code"])

    print("Tool Result:", tool_result)

else:
    print(result.content)


Tool Result: {'customers': [['Harrison', 'Chase'], ['Lang', 'Chain'], ['Dolly', 'Too'], ['Elle', 'Elem'], ['Geoff', 'Fusion'], ['Trance', 'Former'], ['Jen', 'Ayai']], 'sorted_customers': [['Jen', 'Ayai'], ['Lang', 'Chain'], ['Harrison', 'Chase'], ['Elle', 'Elem'], ['Trance', 'Former'], ['Geoff', 'Fusion'], ['Dolly', 'Too']]}


In [23]:
from datetime import date

@tool
def time(time: str) -> str:
    """Returns today's date"""
    return str(date.today())

In [24]:
tools = [calculator, wiki_search, time, python_repl]

llm_with_tools = llm.bind_tools(tools)

In [25]:
result = llm_with_tools.invoke("What's the date today")
if result.tool_calls:
    tool_call = result.tool_calls[0]

    tool_name = tool_call["name"]
    args = tool_call["args"]

    tool_map = {t.name: t for t in tools}

    tool_result = tool_map[tool_name].invoke(args["time"])

    print("Tool Result:", tool_result)

else:
    print(result.content)

Tool Result: 2026-04-01
